In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.environ['API_KEY']

API_KEY

'5fee4d00-94ed-43ce-8b74-975fc923579a'

In [ ]:
"""Fetch up to 1,000 CS2 players per country, across all advertised regions.

Paste this entire file into one Jupyter cell, set API_KEY, and run the cell.
Outputs are saved in a new folder under the notebook kernel working directory.

Uses only Python's standard library (Python 3.9+).
Docs: https://docs.faceit.com/docs/data-api/data/
Country means FACEIT's country field, not verified nationality/residence.
Rankings can change while pages are downloaded; this is not an atomic snapshot.
"""

#API_KEY = "PUT_YOUR_FACEIT_API_KEY_HERE"
GAME = "cs2"
TOP_N = 1000
COUNTRIES = {
    "pk": "Pakistan",
    "ae": "United Arab Emirates",
    "ir": "Iran",
    "bh": "Bahrain",
    "qa": "Qatar",
    "sa": "Saudi Arabia",
    "kw": "Kuwait",
    "om": "Oman",
}

import csv
import json
import math
import os
import time
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import quote, urlencode
from urllib.request import Request, urlopen
from uuid import UUID

BASE_URL = "https://open.faceit.com/data/v4"
PAGE_SIZE = 100
REQUEST_DELAY = 0.5
MAX_ATTEMPTS = 6


def retry_delay(header, attempt):
    """Respect Retry-After seconds or HTTP date; otherwise use backoff."""
    if header:
        try:
            seconds = float(header)
            if math.isfinite(seconds):
                return max(0.0, seconds)
        except ValueError:
            try:
                return max(0.0, parsedate_to_datetime(header).timestamp() - time.time())
            except (ValueError, TypeError, OverflowError):
                pass
    return min(2 ** (attempt + 1), 60)


def get_json(path, api_key, params=None):
    url = BASE_URL + path
    if params:
        url += "?" + urlencode(params)
    request = Request(url, headers={
        "Authorization": "Bearer " + api_key,
        "Accept": "application/json",
    })
    for attempt in range(MAX_ATTEMPTS):
        time.sleep(REQUEST_DELAY)
        try:
            with urlopen(request, timeout=30) as response:
                data = json.load(response)
            if not isinstance(data, dict) or "errors" in data:
                raise RuntimeError("Unexpected FACEIT response for " + path)
            return data
        except HTTPError as exc:
            status = exc.code
            retry_after = exc.headers.get("Retry-After") if exc.headers else None
            exc.close()
            if status in (401, 403):
                raise RuntimeError("FACEIT rejected access. Check your API key and permissions.") from None
            if status != 429 and not 500 <= status < 600:
                raise RuntimeError(f"FACEIT HTTP {status} for {path}; request was not completed.") from None
            reason = f"HTTP {status}"
        except (URLError, TimeoutError, ConnectionError, json.JSONDecodeError) as exc:
            retry_after = None
            reason = type(exc).__name__
        if attempt == MAX_ATTEMPTS - 1:
            raise RuntimeError(f"Request failed after {MAX_ATTEMPTS} attempts: {path} ({reason})")
        delay = retry_delay(retry_after, attempt)
        print(f"  {reason}: retrying in {delay:.1f}s", flush=True)
        time.sleep(delay)


def get_regions(api_key):
    regions = get_json(f"/games/{GAME}", api_key).get("regions")
    if not isinstance(regions, list) or not regions:
        raise RuntimeError("FACEIT did not return CS2 regions. No country results were assumed complete.")
    if any(not isinstance(r, str) or not r.strip() for r in regions):
        raise RuntimeError("Unexpected region format.")
    return list(dict.fromkeys(regions))


def validate_player(item, country):
    if not isinstance(item, dict):
        raise RuntimeError("Ranking entry is not an object.")
    try:
        UUID(item["player_id"])
    except (KeyError, ValueError, TypeError, AttributeError):
        raise RuntimeError("Ranking entry has a missing or invalid player ID.") from None
    if str(item.get("country", "")).lower() != country:
        raise RuntimeError(f"Country filter mismatch: expected {country}, got {item.get('country')!r}.")
    for field in ("faceit_elo", "position"):
        value = item.get(field)
        if type(value) is not int or value < (1 if field == "position" else 0):
            raise RuntimeError(f"Invalid {field} in ranking response.")


def fetch_region(country, region, api_key, target=TOP_N):
    """Read each region's top N country entries; their union contains top N globally."""
    players = {}
    offset = 0
    duplicates = 0
    # A finite guard for an API that ignores pagination or changes continuously.
    max_pages = (target + PAGE_SIZE - 1) // PAGE_SIZE + 10
    for _ in range(max_pages):
        limit = min(PAGE_SIZE, target - len(players))
        page = get_json(
            f"/rankings/games/{GAME}/regions/{quote(region, safe='')}",
            api_key,
            {"country": country, "offset": offset, "limit": limit},
        )
        items = page.get("items")
        if not isinstance(items, list) or len(items) > limit:
            raise RuntimeError("Unexpected ranking page shape.")
        if not items:
            return list(players.values()), duplicates
        before = len(players)
        for item in items:
            validate_player(item, country)
            pid = item["player_id"]
            if pid in players:
                duplicates += 1
            players[pid] = {**item, "region": region}
        print(f"  {country.upper()} / {region}: {len(players)} players", flush=True)
        if len(players) >= target:
            return list(players.values()), duplicates
        if len(players) == before:
            raise RuntimeError(f"Pagination made no progress for {country}/{region}.")
        offset += len(items)
        # Continue even after a short page: stop only on an empty response.
    raise RuntimeError(f"Pagination did not finish for {country}/{region}.")


def merge_players(rows, country, target=TOP_N):
    unique = {}
    for row in rows:
        validate_player(row, country)
        pid = row["player_id"]
        if pid not in unique:
            unique[pid] = {**row, "regions_seen": [row["region"]]}
        else:
            previous = unique[pid]
            regions = sorted(set(previous["regions_seen"] + [row["region"]]))
            # If the leaderboard changed during collection, retain the latest observation.
            unique[pid] = {**row, "regions_seen": regions}
    selected = sorted(unique.values(), key=lambda p: (-p["faceit_elo"], p["player_id"]))[:target]
    return [
        {"_id": p["player_id"], "player_id": p["player_id"],
         "nickname": p.get("nickname", ""), "country": country,
         "country_name": COUNTRIES[country], "selected_rank": rank,
         "faceit_elo": p["faceit_elo"], "game_skill_level": p.get("game_skill_level"),
         "regions_seen": p["regions_seen"], "source_position": p["position"],
         "source_region": p["region"]}
        for rank, p in enumerate(selected, 1)
    ]


def save_json(path, data):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")
    temporary.replace(path)


def main():
    key = API_KEY.strip()
    if not key or key == "PUT_YOUR_FACEIT_API_KEY_HERE":
        key = os.environ.get("FACEIT_API_KEY", "").strip()
    if not key or key == "PUT_YOUR_FACEIT_API_KEY_HERE":
        raise RuntimeError("Put your API key into API_KEY near the top of this script.")
    if key.lower().startswith("bearer "):
        key = key[7:].strip()
    started = datetime.now(timezone.utc)
    output = Path.cwd() / ("faceit_players_" + started.strftime("%Y%m%d_%H%M%S_%f"))
    output.mkdir()
    summary = {"game": GAME, "requested_per_country": TOP_N,
               "started_at": started.isoformat(), "status": "running", "countries": {}}
    save_json(output / "summary.json", summary)
    all_players = []
    try:
        regions = ['EU']
        summary["regions"] = regions
        print("Regions: " + ", ".join(regions), flush=True)
        for country, name in COUNTRIES.items():
            print(f"\nFetching {name}...", flush=True)
            rows = []
            duplicates = 0
            for region in regions:
                region_rows, repeated = fetch_region(country, region, key)
                rows.extend(region_rows)
                duplicates += repeated
            players = merge_players(rows, country)
            save_json(output / f"{country}_players.json", players)
            all_players.extend(players)
            summary["countries"][country] = {
                "name": name, "count": len(players),
                "target_reached": len(players) == TOP_N,
                "duplicate_entries_within_pages": duplicates,
                "duplicate_entries_across_regions": len(rows) - len({r['player_id'] for r in rows}),
            }
            save_json(output / "summary.json", summary)
            print(f"Saved {len(players)} / {TOP_N} for {name}.", flush=True)
        ids = [p["player_id"] for p in all_players]
        if len(ids) != len(set(ids)):
            raise RuntimeError("A player appeared under multiple countries. Country settings may have changed; inspect country files.")
        save_json(output / "all_players.json", all_players)
        save_json(output / "player_ids.json", ids)
        with (output / "all_players.csv").open("w", newline="", encoding="utf-8") as handle:
            fields = list(all_players[0]) if all_players else ["player_id", "country", "faceit_elo"]
            writer = csv.DictWriter(handle, fieldnames=fields)
            writer.writeheader()
            for player in all_players:
                writer.writerow({**player, "regions_seen": ",".join(player["regions_seen"])})
        summary["status"] = "complete"
        summary["total_unique_players"] = len(ids)
        summary["finished_at"] = datetime.now(timezone.utc).isoformat()
        save_json(output / "summary.json", summary)
        print(f"\nDone: {len(ids)} unique players. Files: {output}")
        print("Countries with fewer ranked players keep the available count; see summary.json.")
    except (Exception, KeyboardInterrupt) as exc:
        summary["status"] = "incomplete"
        summary["error"] = str(exc) if not isinstance(exc, KeyboardInterrupt) else "Interrupted by user"
        save_json(output / "summary.json", summary)
        print(f"\nIncomplete run. Completed country files remain in: {output}")
        raise


# Run this cell after setting API_KEY above.
main()



Regions: EU, NA, SA, SEA, OCE

Fetching Pakistan...
  PK / EU: 100 players
  PK / EU: 200 players
  PK / EU: 300 players
  PK / EU: 400 players
  PK / EU: 500 players
  PK / EU: 600 players
  PK / EU: 700 players
  PK / EU: 800 players
  PK / EU: 900 players
  PK / EU: 1000 players
  PK / NA: 72 players

Incomplete run. Completed country files remain in: /media/noir/SSD3/MLE2025/faceit-ai/faceit_players_20260911_144244_318883


RuntimeError: FACEIT HTTP 400 for /rankings/games/cs2/regions/NA; request was not completed.